In [125]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, avg,col,first,when,udf,regexp_replace,to_timestamp
from pyspark.sql.types import StringType, DoubleType
import re
ss = SparkSession.builder.config("spark.jars", "/home/jrodarte/postgresql-42.7.3.jar").getOrCreate()

In [126]:
dfMovimientos = ss.read.format("csv").options(header='true', inferSchema='true', delimiter=',').load("movimientos.csv")
#dfMovimientos.schema

In [127]:
def limpiarImporte(importe):
    return importe.replace("$", "").replace(",", "")


print(limpiarImporte('$0.00'))
limpiarImpoUDF = udf(limpiarImporte, StringType())

0.00


In [128]:
nuevosNombresMov = ['autorizacion', 'feoperacion', 'tipo', 'importe', 'estatus', 'referencia']
nuevosNombresDetMov = ['autorizacion', 'detalle']
dfDetalleMovimiento = dfMovimientos.select('Autorización', 'Detalle')
dfMovimientos = dfMovimientos.selectExpr(
    "`Autorización`",
    "`Fecha operación`",
    "`Tipo`",
    "`Importe`",
    "`Estatus`",
    "`Ref. 1`"
)

In [129]:
dfMovimientos = dfMovimientos.toDF(*nuevosNombresMov)
dfMovimientos = dfMovimientos.withColumn('feoperacion', to_timestamp(col("feoperacion"), "dd/MM/yyyy HH:mm:ss"))
dfMovimientos = dfMovimientos.withColumn('importeS', limpiarImpoUDF(col("importe")))
dfMovimientos = dfMovimientos.withColumn('importe', col("importeS").cast(DoubleType())).drop("importeS")
dfDetalleMovimiento = dfDetalleMovimiento.toDF(*nuevosNombresDetMov)
dfMovimientos.show()
dfMovimientos.schema

+------------+-------------------+-----+-------+--------+--------------------+
|autorizacion|        feoperacion| tipo|importe| estatus|          referencia|
+------------+-------------------+-----+-------+--------+--------------------+
|   103124028|2025-07-31 17:06:19|PAGOS|    0.0|APLICADO|PAGO DE: HARVEY RUIZ|
|   103105326|2025-07-31 16:58:03|PAGOS|   0.01|APLICADO|     PAGO DE: GMORIN|
|   103077947|2025-07-31 10:46:32|PAGOS|  11.17|APLICADO|     PAGO DE: PORUGA|
|   103077946|2025-07-31 10:46:32|PAGOS|    0.0|APLICADO|       RETENCION ISR|
|   103077945|2025-07-31 10:46:32|PAGOS|    0.0|APLICADO|       RETENCION IVA|
|   102992053|2025-07-30 13:46:03|PAGOS|    0.0|APLICADO|       RETENCION ISR|
|   102992052|2025-07-30 13:46:03|PAGOS|    0.0|APLICADO|       RETENCION IVA|
|   102992054|2025-07-30 13:46:03|PAGOS|  13.99|APLICADO|       PAGO DE: MNK@|
|   102975097|2025-07-30 11:01:14|PAGOS|    2.4|APLICADO|     PAGO DE: GMORIN|
|   102975096|2025-07-30 11:01:14|PAGOS|    0.0|APLI

StructType([StructField('autorizacion', IntegerType(), True), StructField('feoperacion', TimestampType(), True), StructField('tipo', StringType(), True), StructField('importe', DoubleType(), True), StructField('estatus', StringType(), True), StructField('referencia', StringType(), True)])

In [130]:
dfMovimientos.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "prestadero.movimientos") \
    .option("user", "jrodarte") \
    .option("password", "roma1993_") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [30]:
ptrDigitos = r"\d\s"
ptrEspacioFinal = r"\s$"

In [31]:
strDetalle = 'Principal: 0.00000 Interes: 0.00000 Impuesto Interes:  0.00000 Moratorios: 0.00281  Impuesto Moratorios: 0.00045 IVA Comisión Moratorios:  0.00000'

In [32]:
coincidencia = re.findall(ptrDigitos, strDetalle)

In [33]:
print(coincidencia)
print(len(coincidencia))

['0 ', '0 ', '0 ', '1 ', '5 ']
5


In [34]:
arrDetalle = re.split(patron, strDetalle)
print(arrDetalle)

['Principal: 0.0000', 'Interes: 0.0000', 'Impuesto Interes:  0.0000', 'Moratorios: 0.0028', ' Impuesto Moratorios: 0.0004', 'IVA Comisión Moratorios:  0.00000']


In [35]:
arrDetalleN = []
j = 0
for i in arrDetalle:    
    var = i+''+coincidencia[j]
    var = re.sub(ptrEspacioFinal, "", var)
    arrDetalleN.append(var)
    j = j + 1
    if len(coincidencia) == j:
        arrDetalleN.append(arrDetalle[j])
        break

In [36]:
print(arrDetalleN)

['Principal: 0.00000', 'Interes: 0.00000', 'Impuesto Interes:  0.00000', 'Moratorios: 0.00281', ' Impuesto Moratorios: 0.00045', 'IVA Comisión Moratorios:  0.00000']
